In [ ]:
pip install matplotlib

In [ ]:
pip install pandas

In [ ]:
pip install --upgrade pip

In [ ]:
pip install seaborn

In [ ]:
pip install openpyxl

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill

# Função para calcular o número triangular fuzzy (mínimo, média, máximo)
def calcular_fuzzy(valores):
    return (np.min(valores), np.mean(valores), np.max(valores))

# Função para defuzzificação usando a média ponderada
# (mínimo + 4*média + máximo) / 6
def defuzzificar(fuzzy):
    return (fuzzy[0] + 4*fuzzy[1] + fuzzy[2]) / 6

# Carregar os dados
file_name = "dataset.xlsx"
df = pd.read_excel(file_name, sheet_name=0)

# Identificar os indicadores
indicadores = df.iloc[:, 0]

# Transformar os dados para cálculos
dados = df.iloc[:, 2:].values

# Aplicar o cálculo fuzzy a cada linha
valores_fuzzy = np.array([calcular_fuzzy(linha) for linha in dados])
valores_defuzzificados = np.array([defuzzificar(fuzzy) for fuzzy in valores_fuzzy])

# Cálculo das estatísticas
medias = valores_defuzzificados
desvios = np.std(dados, axis=1, ddof=1)
cv = (desvios / medias) * 100
minimos = valores_fuzzy[:, 0]
maximos = valores_fuzzy[:, 2]
medianas = np.median(dados, axis=1)

# Criar um DataFrame com os resultados
resultados_df = pd.DataFrame({
    "Indicator": indicadores,
    "Mínimo": minimos,
    "Máximo": maximos,
    "Média Fuzzy": medias,
    "Mediana": medianas,
    "Desvio Padrão": desvios,
    "Coefficient of Variation (%)": cv
})

# Filtrar indicadores com coeficiente de variação menor que 22%
selecionados = resultados_df[resultados_df["Coefficient of Variation (%)"] < 22.00]

# Criar um novo arquivo Excel formatado
output_file = "results.xlsx"
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    resultados_df.to_excel(writer, sheet_name="Resultados", index=False)
    selecionados.to_excel(writer, sheet_name="Selecionados", index=False)

# Melhorar a formatação do Excel
wb = openpyxl.load_workbook(output_file)
ws = wb["Resultados"]
for col in ws.columns:
    for cell in col:
        cell.alignment = Alignment(horizontal='center')
        if cell.row == 1:
            cell.font = Font(bold=True)
            cell.fill = PatternFill(start_color="FFFF99", end_color="FFFF99", fill_type="solid")
wb.save(output_file)

# Configuração de gráficos
sns.set(style="whitegrid")

# Selecionar os 10 indicadores com menor Coeficiente de Variação
top10_cv = resultados_df.sort_values(by="Coefficient of Variation (%)").head(10)

# Obter índices e dados correspondentes
indices_top10 = top10_cv.index
dados_top10 = dados[indices_top10]
indicadores_top10 = indicadores.iloc[indices_top10]

# Criar a pasta Fig, caso ela ainda não exista
Path("Fig").mkdir(exist_ok=True)

# Histograma das médias Fuzzy
graph_file1 = "Fig/histograma_medias_fuzzy.png"
plt.figure(figsize=(10, 5))
sns.histplot(resultados_df["Média Fuzzy"], bins=10, kde=True)
plt.title("Distribuição das Médias Fuzzy dos Indicadores")
plt.xlabel("Média Fuzzy")
plt.ylabel("Frequência")
plt.savefig(graph_file1)

# Gráfico de dispersão do Coeficiente de Variação
graph_file2 = "Fig/dispersao_cv_fuzzy.png"
plt.figure(figsize=(20, 10))
sns.scatterplot(x=resultados_df["Indicator"], y=resultados_df["Coefficient of Variation (%)"], color='blue')
plt.xticks(rotation=90)
#plt.title("Coefficient of Variation by Indicator (Fuzzy)")
plt.xlabel("Indicators")
plt.ylabel("Coefficient of Variation (%)")
plt.axhline(y=22, color='r', linestyle='--', label='22% limit')
plt.legend()
plt.savefig(graph_file2)

# Boxplot para análise da dispersão dos dados
plt.figure(figsize=(12, 6))
sns.boxplot(data=dados_top10)
plt.xticks(ticks=range(len(indicadores_top10)), labels=indicadores_top10, rotation=90)
plt.title("Boxplot dos 10 Indicadores com Menor CV")
plt.tight_layout()
plt.savefig("Fig/boxplot_top10_cv.png")

# Heatmap de correlação
corr_top10 = np.corrcoef(dados_top10)

plt.figure(figsize=(10, 8))
sns.heatmap(corr_top10, annot=True, fmt=".2f", cmap="coolwarm", 
            xticklabels=indicadores_top10, yticklabels=indicadores_top10)
plt.title("Matriz de Correlação - Top 10 Indicadores com Menor CV")
plt.tight_layout()
plt.savefig("Fig/heatmap_top10_cv.png")
